# ステーブルコイン

## ステーブルコインとは

**ステーブルコイン（stablecoin）** は、米ドルなど何らかの資産と価値が連動（ペッグ）するように設計されたトークン。暗号資産の価格変動の大きさを避けつつ、ブロックチェーン上で24時間即時に送金・決済できる。

- DeFiでの取引・貸借の基軸通貨
- 国際送金・決済（特に自国通貨が不安定な国での米ドルの代替）
- 取引所間の資金移動

ステーブルコインの時価総額は2026年時点で約3,000億ドル規模に達し、米国債の大口保有者として既存の金融市場にも影響を持つようになっている。

## 種類

| 種類 | 裏付け | 例 | 特徴 |
| --- | --- | --- | --- |
| 法定通貨担保型 | 発行額と同額の法定通貨や短期国債を発行体が保管 | USDT（Tether）, USDC（Circle）, JPYC | 仕組みは単純だが、準備金が本当にあるかは発行体と監査を信頼するしかない。発行体はアドレスを凍結できる |
| 暗号資産担保型 | 価格変動を見込んで過剰に預けた暗号資産 | DAI / USDS（Sky, 旧MakerDAO）, LUSD（Liquity） | 担保をオンチェーンで確認できるが、設計が複雑で資本効率が悪い |
| アルゴリズム型 | 担保なし（または一部）。供給量の調整でペッグを保つ | UST（Terra, 2022年に崩壊） | 需要が失われると崩壊しやすい |
| 合成ドル型 | 現物の暗号資産と先物のショートを組み合わせてデルタニュートラルにする | USDe（Ethena） | ファンディングレートが利回りの源泉だが、マイナスになるリスクがある |

## 法定通貨担保型

ユーザーが発行体に米ドルを渡すとトークンが発行（ミント）され、トークンを返すと米ドルが払い戻されて（バーン）される。この **償還の約束** と、それに伴う裁定取引がペッグを支える。

- 市場価格が1ドルを下回る → 安く買って発行体で1ドルに償還すれば儲かる → 買いが入って価格が戻る
- 市場価格が1ドルを上回る → 発行体で1ドルで発行して市場で売れば儲かる → 売りが入って価格が戻る

ただし、償還できるのは発行体と契約した事業者に限られることが多く、準備金の透明性や、規制当局・発行体によるアドレスの凍結が論点になる。2023年3月には、USDCの準備金の一部がシリコンバレー銀行の破綻で引き出せなくなり、USDCが一時0.87ドル付近まで下落した。

## 暗号資産担保型：DAIの仕組み

**DAI** はMakerDAO（2024年にSkyにリブランド、新トークンUSDS）が2017年に発行を始めた、最初の実用的な暗号資産担保型ステーブルコイン。ここでは仕組みが比較的単純な初期の単一担保型（ETHのみを担保とする、通称Sai）を説明する。

### 登場人物

| 役割 | 内容 |
| --- | --- |
| ユーザー | 自分専用の金庫（CDP, Collateralized Debt Position）にETHを預け、それを担保にDAIを発行する |
| キーパー | 担保率を下回った金庫を見つけて清算する。清算すると儲かるインセンティブがある |
| オラクル | ETHの市場価格をプロトコルに提供する |
| MKR保有者 | ガバナンス投票で金利などのパラメーターを決める |

### 担保の形：ETH → WETH → PETH

Saiでは、ETHをそのまま金庫に入れるのではなく、2段階の変換を経て担保にする。

```mermaid
flowchart LR
    ETH["ETH"] -->|"WETHコントラクトに預ける"| WETH["WETH<br/>（ERC-20版のETH、常に1:1）"]
    WETH -->|"MakerDAOに預ける"| PETH["PETH<br/>（担保プールの持ち分）"]
    PETH -->|"lock"| CDP["金庫（CDP）"]
    CDP -->|"draw"| DAI["DAI"]
```

- **WETH（Wrapped Ether）**：ETHはプロトコルのネイティブ通貨でERC-20のインターフェースを持たないため、コントラクトで扱いやすいようにERC-20として包んだもの。WETHのコントラクトにETHを預けると同量のWETHが発行され、返せばETHに戻る。運営者のいない単純なコントラクトなので、ほぼETHそのものとして扱える（Uniswapなど多くのDeFiでも使われる）
- **PETH（Pooled Ether）**：MakerDAOに預けられたWETH全体（担保プール）に対する持ち分を表すトークン。交換比率は1:1に固定されておらず、**プール内のWETHの量 ÷ PETHの発行量** で決まる

PETHの交換比率は、後述する清算の損益に応じて変動する。つまりPETHで担保を預けることは、清算の利益と損失を担保提供者全員で分け合う保険のプールに出資するようなものだった。PETHは2019年の多担保化で廃止された。

### 発行と返済

DAIの発行は、ETH（PETH）を担保にした借り入れに近い。

| 操作 | 内容 |
| --- | --- |
| lock | 担保（PETH）を金庫に預ける |
| draw | DAIを新規発行する。担保率が150%を下回る量は発行できない |
| wipe | DAIを返済する。返済されたDAIはバーンされる |
| free | 担保を引き出す。担保率が150%を下回る量は引き出せない |

ETHの価格変動に備えて、DAIの発行には **150%以上の担保** が必要。つまり、担保のETHのドル建て価値の2/3までしかDAIを発行できない。

### ペッグの維持：安定化手数料

返済時には、借りていた期間に応じた **安定化手数料（stability fee）** を払う。この料率（金利）はMKR保有者の投票で決められ、金融政策のように使われる。

- DAIの需要が高い（1 DAI > 1 USD）→ 金利を下げてDAIの発行を促す（金融緩和）
- DAIの需要が低い（1 DAI < 1 USD）→ 金利を上げてDAIの返済を促す（金融引き締め）

後に、DAIを預けると利息が得られる **DAI Savings Rate（DSR）** も導入され、需要側からもペッグを調整できるようになった。

### 清算

ETHが値下がりして担保率が150%を割った金庫は、**誰でも（キーパーとして）清算できる**。

1. 金庫の担保が没収され、発行済みDAI × 1.13相当のETHが清算用のコントラクトに移される（ユーザーへの13%のペナルティ）
2. 清算コントラクトは、没収したETHを市場価格より3%安くDAIで販売し、市場からDAIを回収する
3. 回収したDAIと、清算された金庫の未回収分のDAIを相殺してバーンする

キーパーにとっては、ETHを割安に買える裁定機会になっている。

清算の損益はPETHの交換比率に反映される。

- **清算で余りが出た場合**（13%のペナルティ分など）：余ったDAIでPETHを買い戻してバーンする（boom）。PETHの量が減るので、1 PETHあたりのWETHが増え、担保提供者全員が得をする
- **担保の急落で不足が出た場合**：最終手段としてPETHを新規発行して売却し、不足分を埋める（bust）。PETHが希薄化して1 PETHあたりのWETHが減り、担保提供者全員が損をする

多担保化後は、不足分はMKRを新規発行して売ることで埋める仕組みになり、損失の負担者がMKR保有者に変わった。

2020年3月のコロナショックでは、ETHの急落とEthereumの混雑で清算オークションがほぼゼロ円で落札される事態が起き、不足分を埋めるためにMKRが発行された。

### 多担保化とRWA

2019年11月に複数担保型（MCD, Multi-Collateral DAI）に移行し、ETH以外のトークンも担保にできるようになった。その後、ペッグを安定させるためにUSDCを大量に担保として取り込み、さらに米国債などの **RWA（現実資産）** も担保に加えたことで、「分散型なのに中央集権的な資産に依存している」という議論も起きている。

## アルゴリズム型：Terra/USTの崩壊

**UST** は、Terraチェーンのネイティブトークン **LUNA** との交換によってペッグを保とうとしたアルゴリズム型ステーブルコイン。

- プロトコルは常に「1 USTと1ドル分のLUNAを交換する」
- UST < 1ドルなら、USTを安く買ってプロトコルで1ドル分のLUNAに替えれば儲かる → USTの供給が減って価格が戻る（はず）

関連するレンディングプロトコルAnchorがUSTの預金に約20%の利回りを提供していたことで急成長したが、2022年5月、大口のUST売却をきっかけにペッグが外れると、

UST売り → LUNAの大量発行 → LUNAの暴落 → USTの信用低下 → さらなるUST売り

という **デススパイラル** に陥り、数日でLUNAとUSTの価値はほぼゼロになった（約400億ドル規模が消失）。裏付け資産のない仕組みは、信用が失われたときに歯止めがかからないことを示した。

## 規制

2022年以降、各国でステーブルコインを法的に位置づける動きが進んでいる。

| 地域 | 規制 | 概要 |
| --- | --- | --- |
| 日本 | 改正資金決済法（2023年施行） | 法定通貨建てのステーブルコインを「電子決済手段」と定義。発行者は銀行、資金移動業者、信託会社に限定。2025年にJPYCが資金移動業者として円建てステーブルコインの発行を開始 |
| EU | MiCA（2024年全面適用） | ステーブルコイン（EMT, ART）の発行者にライセンスと準備金の要件を課す |
| 米国 | GENIUS Act（2025年成立） | 支払用ステーブルコインの発行者に、現金や短期米国債などによる1:1の準備金と定期的な開示を義務付ける |

## 参考文献

- MakerDAO. [The Dai Stablecoin System (Sai Whitepaper)](https://makerdao.com/en/whitepaper/sai/)
- Klages-Mundt, A. et al. (2020). [Stablecoins 2.0: Economic Foundations and Risk-based Models](https://arxiv.org/abs/2006.12388). ACM AFT.
- Liu, J., Makarov, I. & Schoar, A. (2023). [Anatomy of a Run: The Terra Luna Crash](https://www.nber.org/papers/w31160). NBER Working Paper.
- [rwa.xyz Stablecoins](https://app.rwa.xyz/stablecoins)